# 04 — Event Emitters & Streams

The EventEmitter pattern and Streams are core to Node.js architecture. Interviewers use these to test if you truly understand how Node.js works under the hood.

---

## Table of Contents
1. The EventEmitter Pattern
2. Building Custom EventEmitters
3. Buffers
4. Streams — Types & Use Cases
5. Readable Streams
6. Writable Streams
7. Transform Streams
8. Piping & Pipeline
9. Backpressure
10. Interview Questions

---
## 1. The EventEmitter Pattern

Node.js is built on an **event-driven architecture**. The `EventEmitter` class is the foundation — many core modules (`http`, `fs`, `net`, `stream`) inherit from it.

### Core concept:
- **Emitter** fires named events
- **Listeners** (callbacks) subscribe to those events
- It's the **Observer pattern** (publish/subscribe)

In [ ]:
const EventEmitter = require('events');

const emitter = new EventEmitter();

// Register a listener
emitter.on('greet', (name) => {
    console.log(`Hello, ${name}!`);
});

// Register another listener for the same event
emitter.on('greet', (name) => {
    console.log(`Welcome aboard, ${name}!`);
});

// Emit the event — both listeners fire in order
emitter.emit('greet', 'Alice');

In [ ]:
const EventEmitter = require('events');
const emitter = new EventEmitter();

// .once() — listener fires only once, then auto-removes
emitter.once('connect', () => {
    console.log('Connected (fires once)');
});
emitter.emit('connect'); // fires
emitter.emit('connect'); // nothing happens

// Remove a listener
function onData(data) { console.log('Data:', data); }
emitter.on('data', onData);
emitter.off('data', onData); // or: emitter.removeListener('data', onData)
emitter.emit('data', 'test'); // nothing — listener removed

// Useful methods
console.log('Listener count for "data":', emitter.listenerCount('data'));
console.log('Event names:', emitter.eventNames());

In [ ]:
const EventEmitter = require('events');

// 'error' event — SPECIAL: if no listener, it throws and crashes!
const emitter = new EventEmitter();

// Always add error handlers!
emitter.on('error', (err) => {
    console.error('Caught error:', err.message);
});

emitter.emit('error', new Error('Something broke'));
console.log('Process still alive because we handled the error');

---
## 2. Building Custom EventEmitters

A common interview task: "Build a class that extends EventEmitter."

In [ ]:
const EventEmitter = require('events');

class JobQueue extends EventEmitter {
    constructor() {
        super();
        this.jobs = [];
    }

    addJob(job) {
        this.jobs.push(job);
        this.emit('jobAdded', job);
    }

    async processAll() {
        this.emit('start', this.jobs.length);
        for (const job of this.jobs) {
            try {
                const result = await job();
                this.emit('jobComplete', result);
            } catch (err) {
                this.emit('jobError', err);
            }
        }
        this.jobs = [];
        this.emit('done');
    }
}

const queue = new JobQueue();
queue.on('jobAdded', (job) => console.log('Job added to queue'));
queue.on('start', (count) => console.log(`Processing ${count} jobs...`));
queue.on('jobComplete', (result) => console.log('  Completed:', result));
queue.on('jobError', (err) => console.error('  Failed:', err.message));
queue.on('done', () => console.log('All jobs processed!'));

queue.addJob(async () => { await new Promise(r => setTimeout(r, 50)); return 'Job 1 result'; });
queue.addJob(async () => { await new Promise(r => setTimeout(r, 30)); return 'Job 2 result'; });
queue.addJob(async () => { throw new Error('Job 3 failed'); });

queue.processAll();

---
## 3. Buffers

Buffers represent **fixed-length sequences of bytes**. They're essential for working with binary data (files, network packets, images).

Buffers exist because JavaScript strings are UTF-16 encoded, but I/O operations work with raw bytes.

In [ ]:
// Creating buffers
const buf1 = Buffer.alloc(10);                    // 10 zero-filled bytes
const buf2 = Buffer.from('Hello, Node.js!');      // From string
const buf3 = Buffer.from([72, 101, 108, 108, 111]); // From byte array

console.log('buf1:', buf1);              // <Buffer 00 00 00 00 ...>
console.log('buf2:', buf2);              // <Buffer 48 65 6c 6c 6f ...>
console.log('buf3:', buf3.toString());   // 'Hello'

// Buffer properties
console.log('buf2 length (bytes):', buf2.length);
console.log('buf2 as string:', buf2.toString('utf8'));
console.log('buf2 as hex:', buf2.toString('hex'));
console.log('buf2 as base64:', buf2.toString('base64'));

In [ ]:
// Buffer operations
const buf = Buffer.alloc(256);

// Write to buffer
const bytesWritten = buf.write('Node.js is awesome');
console.log('Bytes written:', bytesWritten);
console.log('Content:', buf.toString('utf8', 0, bytesWritten));

// Slicing (creates a VIEW, not a copy!)
const slice = buf.slice(0, 7);
console.log('Slice:', slice.toString()); // 'Node.js'

// Comparing
const a = Buffer.from('abc');
const b = Buffer.from('abc');
const c = Buffer.from('def');
console.log('a equals b:', a.equals(b));     // true
console.log('a equals c:', a.equals(c));     // false
console.log('a compare c:', a.compare(c));   // -1 (a < c)

---
## 4. Streams — Overview

Streams let you process data **piece by piece** instead of loading it all into memory. Essential for handling large files, network data, or any data source that's too big to fit in RAM.

### The 4 types of streams:

| Type | Description | Example |
|------|-----------|--------|
| **Readable** | Source of data | `fs.createReadStream()`, `http.IncomingMessage` |
| **Writable** | Destination for data | `fs.createWriteStream()`, `http.ServerResponse` |
| **Duplex** | Both readable and writable | `net.Socket`, `WebSocket` |
| **Transform** | Duplex that modifies data | `zlib.createGzip()`, `crypto.createCipher()` |

### Why streams matter:
```
Without streams: Read 2GB file → Store entire 2GB in memory → Process → Write
With streams:    Read 64KB chunk → Process → Write → Read next chunk → ...
```

In [ ]:
// Memory comparison: reading a large file
const fs = require('fs');

// WITHOUT streams — loads entire file into memory
// const data = fs.readFileSync('huge-file.csv', 'utf8'); // ← 2GB in RAM!

// WITH streams — processes chunk by chunk (~64KB at a time)
// const stream = fs.createReadStream('huge-file.csv', 'utf8');
// stream.on('data', (chunk) => { /* process 64KB chunk */ });

console.log('Streams process data in chunks, keeping memory usage constant');
console.log('This is why Node.js can handle files larger than available RAM');

---
## 5. Readable Streams

In [ ]:
const { Readable } = require('stream');

// Creating a custom Readable stream
class Counter extends Readable {
    constructor(options) {
        super(options);
        this.current = 1;
        this.max = 5;
    }

    _read() {
        if (this.current <= this.max) {
            this.push(`Count: ${this.current}\n`);
            this.current++;
        } else {
            this.push(null); // signals end of stream
        }
    }
}

const counter = new Counter();

// Consuming with 'data' event (flowing mode)
counter.on('data', (chunk) => process.stdout.write(chunk.toString()));
counter.on('end', () => console.log('Stream ended'));

In [ ]:
const { Readable } = require('stream');

// Readable.from() — create a stream from an iterable
async function* generateNumbers() {
    for (let i = 1; i <= 5; i++) {
        yield `Number ${i}\n`;
    }
}

const stream = Readable.from(generateNumbers());

// Consuming with for-await-of (modern approach)
(async () => {
    for await (const chunk of stream) {
        process.stdout.write(chunk);
    }
    console.log('Async iteration complete');
})();

### Readable Stream Modes:
- **Flowing mode** — data is read automatically and provided via events (`'data'`)
- **Paused mode** — you must explicitly call `.read()` to get chunks

Switch to flowing mode by: adding a `'data'` listener, calling `.resume()`, or `.pipe()`  
Switch to paused mode by: calling `.pause()`, or removing all `'data'` listeners

---
## 6. Writable Streams

In [ ]:
const { Writable } = require('stream');

// Custom writable stream (e.g., collect data into an array)
class ArrayWriter extends Writable {
    constructor(options) {
        super(options);
        this.data = [];
    }

    _write(chunk, encoding, callback) {
        this.data.push(chunk.toString());
        console.log(`  Written: ${chunk.toString().trim()}`);
        callback(); // MUST call callback to signal you're ready for more data
    }
}

const writer = new ArrayWriter();
writer.write('Line 1\n');
writer.write('Line 2\n');
writer.write('Line 3\n');
writer.end('Final line\n'); // end() can also write last chunk

writer.on('finish', () => {
    console.log('All data:', writer.data);
});

---
## 7. Transform Streams

In [ ]:
const { Transform } = require('stream');

// Transform stream — modifies data as it passes through
class UpperCase extends Transform {
    _transform(chunk, encoding, callback) {
        this.push(chunk.toString().toUpperCase());
        callback();
    }
}

const upper = new UpperCase();
upper.on('data', (chunk) => console.log('Transformed:', chunk.toString()));

upper.write('hello ');
upper.write('world');
upper.end();

---
## 8. Piping & Pipeline

Piping connects a Readable to a Writable, handling backpressure automatically.

In [ ]:
const { Readable, Transform, Writable, pipeline } = require('stream');
const { promisify } = require('util');

// .pipe() — simple but no error handling
// readable.pipe(transform).pipe(writable);

// pipeline() — preferred! Handles errors and cleanup
const pipelineAsync = promisify(pipeline);

const source = Readable.from(['hello ', 'world ', 'streams!']);

const toUpper = new Transform({
    transform(chunk, enc, cb) {
        cb(null, chunk.toString().toUpperCase());
    }
});

const collector = [];
const sink = new Writable({
    write(chunk, enc, cb) {
        collector.push(chunk.toString());
        cb();
    }
});

(async () => {
    await pipelineAsync(source, toUpper, sink);
    console.log('Pipeline result:', collector.join(''));
})();

### `.pipe()` vs `pipeline()`:

| Feature | `.pipe()` | `pipeline()` |
|---------|----------|-------------|
| Error handling | Manual (errors can be swallowed) | Automatic (propagates errors) |
| Cleanup | Must manually destroy streams | Auto-destroys all streams on error |
| Callback/Promise | No | Yes |
| Recommended | Legacy code | New code |

> **Interview Tip:** Always recommend `pipeline()` over `.pipe()`. It's the modern, safe approach.

---
## 9. Backpressure

**Backpressure** occurs when data is being produced faster than it can be consumed.

Example: reading a file at 1GB/s but writing to a network at 100MB/s.

### How Node.js handles it:
1. `.write()` returns `false` when the internal buffer is full
2. The producer should pause
3. The `'drain'` event fires when the buffer is ready for more data

```javascript
// Manual backpressure handling:
readable.on('data', (chunk) => {
    const canContinue = writable.write(chunk);
    if (!canContinue) {
        readable.pause();  // Stop reading!
        writable.once('drain', () => {
            readable.resume(); // Buffer drained, continue
        });
    }
});
```

> **Good news:** `.pipe()` and `pipeline()` handle backpressure automatically.

---
## 10. Interview Questions & Answers

### Q1: What is the EventEmitter pattern?
**A:** It's the Observer/Pub-Sub pattern implemented in Node.js. Objects emit named events that cause registered listener functions to be called. Many core Node.js modules (http, fs, streams) are built on EventEmitter.

### Q2: What happens if you emit an `'error'` event with no listener?
**A:** Node.js throws the error and crashes the process. This is special behavior unique to the `'error'` event. Always register error handlers on EventEmitters.

### Q3: What are Streams and why are they useful?
**A:** Streams process data in chunks rather than loading everything into memory. They're essential for handling large files, real-time data, and network communications. A 10GB file can be processed with constant memory usage (~64KB) using streams.

### Q4: What is backpressure?
**A:** Backpressure occurs when a writable stream can't consume data as fast as a readable stream produces it. Node.js streams handle this by returning `false` from `.write()` when the buffer is full, and emitting `'drain'` when ready. `pipeline()` handles this automatically.

### Q5: What's the difference between `.pipe()` and `pipeline()`?
**A:** `pipeline()` is the modern alternative that properly handles errors (auto-destroys streams on failure) and supports a callback/promise for completion. `.pipe()` can silently swallow errors and leak resources.

### Q6: Name the 4 types of streams.
**A:** Readable (data source), Writable (data destination), Duplex (both readable and writable, like a TCP socket), and Transform (Duplex that modifies data passing through, like compression).

### Q7: What is a Buffer?
**A:** A Buffer is a fixed-length sequence of bytes, used to handle binary data in Node.js. Since JavaScript strings are UTF-16, Buffers bridge the gap between JS and raw binary I/O (files, network packets, etc.).